In [ ]:

import sqlite3

# Connect to database
conn = sqlite3.connect("ecommerce.db")
cursor = conn.cursor()

# Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    price REAL NOT NULL,
    stock INTEGER NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT,
    quantity INTEGER,
    total REAL
)
""")

conn.commit()

# Add sample products only once
cursor.execute("SELECT COUNT(*) FROM products")
if cursor.fetchone()[0] == 0:
    sample_products = [
        ("Laptop", 45000, 10),
        ("Mobile Phone", 15000, 20),
        ("Headphones", 1200, 30),
        ("Smart Watch", 2500, 15),
        ("Keyboard", 800, 25)
    ]
    cursor.executemany(
        "INSERT INTO products (name, price, stock) VALUES (?, ?, ?)",
        sample_products
    )
    conn.commit()


# Display products
def display_products():
    cursor.execute("SELECT * FROM products")
    products = cursor.fetchall()

    print("\n========== AVAILABLE PRODUCTS ==========")
    print("ID\tName\t\tPrice\t\tStock")
    print("----------------------------------------")

    for p in products:
        print(f"{p[0]}\t{p[1]}\t\t₹{p[2]}\t\t{p[3]}")


# Add product
def add_product():
    name = input("Enter product name: ")
    price = float(input("Enter product price: "))
    stock = int(input("Enter stock quantity: "))

    if price <= 0 or stock < 0:
        print("Invalid price or stock!")
        return

    cursor.execute(
        "INSERT INTO products (name, price, stock) VALUES (?, ?, ?)",
        (name, price, stock)
    )
    conn.commit()
    print("Product added successfully!")


# Search product
def search_product():
    name = input("Enter product name to search: ")

    cursor.execute(
        "SELECT * FROM products WHERE name LIKE ?",
        ('%' + name + '%',)
    )

    results = cursor.fetchall()

    if results:
        for p in results:
            print(
                f"ID: {p[0]} | Name: {p[1]} | "
                f"Price: ₹{p[2]} | Stock: {p[3]}"
            )
    else:
        print("Product not found!")


# Purchase product
def purchase_product():
    display_products()

    try:
        product_id = int(input("\nEnter product ID: "))
        quantity = int(input("Enter quantity: "))

        if quantity <= 0:
            print("Quantity must be greater than zero!")
            return

        cursor.execute(
            "SELECT * FROM products WHERE id = ?",
            (product_id,)
        )
        product = cursor.fetchone()

        if product is None:
            print("Invalid product ID!")
            return

        if quantity > product[3]:
            print("Sorry! Insufficient stock.")
            return

        total = product[2] * quantity

        print("\n========== YOUR BILL ==========")
        print("Product:", product[1])
        print("Quantity:", quantity)
        print("Price: ₹", product[2])
        print("Total Amount: ₹", total)

        confirm = input("Confirm purchase? (yes/no): ").lower()

        if confirm == "yes":
            cursor.execute(
                "UPDATE products SET stock = stock - ? WHERE id = ?",
                (quantity, product_id)
            )

            cursor.execute("""
                INSERT INTO orders
                (product_name, quantity, total)
                VALUES (?, ?, ?)
            """, (product[1], quantity, total))

            conn.commit()
            print("Order placed successfully!")

        else:
            print("Purchase cancelled.")

    except ValueError:
        print("Please enter valid numbers!")


# View order history
def order_history():
    cursor.execute("SELECT * FROM orders")
    orders = cursor.fetchall()

    print("\n========== ORDER HISTORY ==========")

    if not orders:
        print("No orders found.")
        return

    for order in orders:
        print(
            f"Order ID: {order[0]} | "
            f"Product: {order[1]} | "
            f"Quantity: {order[2]} | "
            f"Total: ₹{order[3]}"
        )


# Main menu
def main():
    while True:
        print("\n====================================")
        print("      E-COMMERCE MANAGEMENT SYSTEM")
        print("====================================")
        print("1. Display Products")
        print("2. Add New Product")
        print("3. Search Product")
        print("4. Purchase Product")
        print("5. View Order History")
        print("6. Exit")

        choice = input("\nEnter your choice: ")

        if choice == "1":
            display_products()

        elif choice == "2":
            add_product()

        elif choice == "3":
            search_product()

        elif choice == "4":
            purchase_product()

        elif choice == "5":
            order_history()

        elif choice == "6":
            print("Thank you for using our E-Commerce System!")
            break

        else:
            print("Invalid choice! Please try again.")


if __name__ == "__main__":
    try:
        main()
    finally:
        conn.close()



      E-COMMERCE MANAGEMENT SYSTEM
1. Display Products
2. Add New Product
3. Search Product
4. Purchase Product
5. View Order History
6. Exit

Enter your choice: 1

========== AVAILABLE PRODUCTS ==========
ID	Name		Price		Stock
----------------------------------------
1	Laptop		₹45000.0		10
2	Mobile Phone		₹15000.0		20
3	Headphones		₹1200.0		30
4	Smart Watch		₹2500.0		15
5	Keyboard		₹800.0		25
6	laptop		₹45000.0		10

      E-COMMERCE MANAGEMENT SYSTEM
1. Display Products
2. Add New Product
3. Search Product
4. Purchase Product
5. View Order History
6. Exit

Enter your choice: 2
Enter product name: mouse
Enter product price: 1000
Enter stock quantity: 10
Product added successfully!

      E-COMMERCE MANAGEMENT SYSTEM
1. Display Products
2. Add New Product
3. Search Product
4. Purchase Product
5. View Order History
6. Exit

Enter your choice: 1

========== AVAILABLE PRODUCTS ==========
ID	Name		Price		Stock
----------------------------------------
1	Laptop		₹45000.0		10
2	Mobile Phone		